# Notebook zu: "Kryptographie Entschlüsselt": Moderne Blockchiffren

© 2001–2026 Gerald Teschl (http://www.mat.univie.ac.at/~gerald/) und Susanne Teschl (http://staff.technikum-wien.at/~teschl/)

Dieses [Python](https://python.org)-Notebook ist als Ergänzung zu unserem [Springer](https://www.springer.de/)-Lehrbuch "[Kryptographie Entschlüsselt](https://www.mat.univie.ac.at/~gerald/ftp/book-kry/)" gedacht. Dort finden Sie auch weitere Erklärungen.

Der Code kann bequem von [PyPI](https://pypi.org/project/kryptools/) installiert werden.

## (S)DES

DES und Varianten verwenden als Hauptoperation die Vertauschung von Bits. Dazu bietet es sich an, die Blöcke zur Verarbeitung in eine Liste der Bits zu verwandeln:

In [1]:
def int2list(x: int, length:int = 8) -> list:
    "Convert an integer or character to a list of binary digits."
    if not isinstance(x, int):
        x = ord(x)
    return [int(d) for d in str(format(x, "0" + str(length) + "b"))]

def list2int(a: list) -> int:
    "Convert a list of binary digits to an integer."
    s = 0
    for i, d in enumerate(reversed(a)):
        if d:
            s += 1 << i
    return s

x = 123
a = int2list(x)
assert list2int(a) == x
a

[0, 1, 1, 1, 1, 0, 1, 1]

Nun können wir (z. B.) die Eingangspermutation des S-DES wie folgt implementieren:

In [2]:
def permute(a: list[int], permutation: tuple[int]) -> list[int]:
    "Apply a permutation to a list."
    return [ a[i-1] for i in permutation]

IP = (2, 6, 3, 1, 4, 8, 5, 7)
permute(a, IP)

[1, 0, 1, 0, 1, 1, 1, 1]

Eine zyklische Verschiebung nach links erreichen wir mit:

In [3]:
def rotate_left(l: list, n: int = 1) -> list:
    "Rotate a list cyclically n places to the left."
    n = n % len(l)
    return l[n:] + l[:n]

rotate_left(a, 2)

[1, 1, 1, 0, 1, 1, 0, 1]

Ein negativer Wert verschiebt nach rechts:

In [4]:
rotate_left(a, -1)

[1, 0, 1, 1, 1, 1, 0, 1]

Damit haben wir alle Bestandteile für die Schlüsselerzeugung:

In [5]:
def sdes_key_schedule(k: list) -> list[list]:
    "SDES key schedule, returning the round keys."
    IP = (3, 5, 2, 7, 4, 10, 1, 9, 8, 6) # PC-1 initial permutation
    CP = (6, 3, 7, 4, 8, 5, 10, 9)  # PC-2 compression permutation for selection of key bits
    ROT = (1, 2)  # schedule of left shift

    key_len2 = len(IP) // 2
    keys = []
    k = permute(k, IP)  # initial permutation
    for i in range(len(ROT)):
        k = rotate_left(k[:key_len2], ROT[i]) + rotate_left(k[key_len2:], ROT[i])
        keys.append(permute(k, CP))
    return keys

key = int2list(34, 10)
keys = sdes_key_schedule(key)
keys

[[0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 1, 0]]

Die gleiche Funktion erzeugt uns auch die Rundenschlüssel für den DES, wenn wir `IP`, `CP` und `ROT` durch die entsprechenden Werte für DES ersetzen.

Unser Paket bietet dafür eine Klasse `FeistelKeySchedule` an, von der die Schlüsselerzeugung durch die Angabe von `IP`, `CP` und `ROT` abgeleitet werden kann. Bei der Initialisierung mit einem Schlüssel wird eine Liste der Bits erzeugt und `IP` angewandt. Die Methode `next` liefert den nächsten Schlüssel. Eine Darstellung der Klasse zeigt den aktuellen Zustand.
Unterklassen `SDESKeySchedule` und `DESKeySchedule` mit den entsprechenden Werten für SDES und DES sind vordefiniert.

In [6]:
from kryptools import SDESKeySchedule

ks = SDESKeySchedule(34)
print("Status: ",ks)
print("Schlüssel 1: ", ks.next())
print("Status: ",ks)
print("Schlüssel 2: ", ks.next())
print("Status: ",ks)

Status:  [0, 1, 0, 0, 0, 0, 0, 1, 0, 0]
Schlüssel 1:  [0, 0, 1, 0, 0, 0, 0, 0]
Status:  [1, 0, 0, 0, 0, 0, 1, 0, 0, 0]
Schlüssel 2:  [0, 0, 0, 1, 0, 0, 1, 0]
Status:  [0, 0, 0, 1, 0, 0, 0, 0, 0, 1]


Analog können wir die weiteren Bestandteile des (S)DES implementieren. Unser Paket bietet dafür eine Klasse `FeistelCipher` an. Diese wird mit einem Schlüssel initialisiert und danach kann durch Aufruf ein Klartextblock als Status geladen werden. Nun können die einzelnen Bestandteile `ip()` (initial permutation), `theta()` (Vertauschen der Hälften), `round(key)` (Rundenfunktion mit Rundenschlüssel `key`) angewendet werden.

In [7]:
from kryptools import SDESCipher

sdes = SDESCipher(34)
sdes(b'\x01')
sdes.ip()
print("IP:     ", sdes)
sdes.round(sdes.keys[0])
print("Runde 1:", sdes)
sdes.round(sdes.keys[1], last=True) # we don not switch halves in the last round
print("Runde 2:", sdes)
sdes.fp()
print("FP:     ", sdes)
int(sdes)

IP:      00000100
Runde 1: 01001110
Runde 2: 00111110
FP:      10111001


185

Natürlich ist es auch möglich, auf einmal zu verschlüsseln:

In [8]:
sdes.encrypt(b'\x01')

b'\xb9'

## DES

Das Paket stellt auf die gleiche Weise auch den vollen DES zur Verfügung. Hier zum Beispiel die Testvektoren aus dem [Handbook of Applied Cryptography](https://cacr.uwaterloo.ca/hac/) (Example 7.86):

In [9]:
from kryptools import DESCipher

k = 0x0123456789ABCDEF.to_bytes(8, 'big')
x = 0x4E6F772069732074.to_bytes(8, 'big')
y = 0x3FA40E8A984D4815.to_bytes(8, 'big')

des = DESCipher(k)
des.encrypt(x)
print(des)
assert bytes(des) == y
assert des.decrypt() == x

3fa40e8a984d4815


Warnung: Sinn der Funktionen aus dem Paket ist es, Zugriff auf die Einzeloperationen zu geben, um damit experimentieren zu können. Für den praktischen Einsatz sind diese ungeeignet!

## AES

Wir verwenden wieder unsere Klassen, um die Operationen im zugehörigen Polynomring durchzuführen. Außerdem verwenden wir zwei Funktionen, die Bytes in Polynome umwandeln und umgekehrt:

In [10]:
from kryptools import Zmod, Poly

Z_2 = Zmod(2)
aes = [1, 1, 0, 1, 1, 0, 0, 0, 1]  # x^8 + x^4 + x^3 + x + 1


def PolyAES(c: list) -> "Poly":
    return Poly(c, ring=Z_2, modulus=aes)

def byte2poly(x: int, n: int = 8) -> "Poly":
    "Convert an integer to a Polynomial."
    return PolyAES(list(reversed([int(d) for d in str(format(x, "0" + str(n) + "b"))])))


def poly2byte(p: "Poly") -> int:
    "Convert a polynomial to the corresponding integer."
    s = 0
    for i in reversed(p.coeff):
        s = s * 2 + int(i)
    return s


poly2byte(byte2poly(2).inv())

141

Damit können wir leicht die S-Box implementieren:

In [11]:
from kryptools import Matrix, zeros

def rotate(l: list, n: int) -> list:
    "Rotate a list cyclically n places to the left."
    n %= len(l)
    return l[n:] + l[:n]

aes_A = Matrix([ rotate([1, 0, 0, 0, 1, 1, 1, 1], -n) for n in range(8) ], ring=Z_2)
aes_A_inv = aes_A.inv()
aes_b = Matrix([1, 1, 0, 0, 0, 1, 1, 0], ring=Z_2)

def aes_sbox(x: Poly) -> Poly:
    "AES S-box."
    if x:
        res = x.inv().coeff
        res += [0] * (8 - len(res))
    else:
        res = [0] * 8
    return PolyAES(aes_A * Matrix(res) + aes_b)

Matrix([[ format(poly2byte(aes_sbox(byte2poly(i + 16 * j))), '02x') for i in range(16)] for j in range(16)])

[ 63, 7c, 77, 7b, f2, 6b, 6f, c5, 30, 01, 67, 2b, fe, d7, ab, 76 ]
[ ca, 82, c9, 7d, fa, 59, 47, f0, ad, d4, a2, af, 9c, a4, 72, c0 ]
[ b7, fd, 93, 26, 36, 3f, f7, cc, 34, a5, e5, f1, 71, d8, 31, 15 ]
[ 04, c7, 23, c3, 18, 96, 05, 9a, 07, 12, 80, e2, eb, 27, b2, 75 ]
[ 09, 83, 2c, 1a, 1b, 6e, 5a, a0, 52, 3b, d6, b3, 29, e3, 2f, 84 ]
[ 53, d1, 00, ed, 20, fc, b1, 5b, 6a, cb, be, 39, 4a, 4c, 58, cf ]
[ d0, ef, aa, fb, 43, 4d, 33, 85, 45, f9, 02, 7f, 50, 3c, 9f, a8 ]
[ 51, a3, 40, 8f, 92, 9d, 38, f5, bc, b6, da, 21, 10, ff, f3, d2 ]
[ cd, 0c, 13, ec, 5f, 97, 44, 17, c4, a7, 7e, 3d, 64, 5d, 19, 73 ]
[ 60, 81, 4f, dc, 22, 2a, 90, 88, 46, ee, b8, 14, de, 5e, 0b, db ]
[ e0, 32, 3a, 0a, 49, 06, 24, 5c, c2, d3, ac, 62, 91, 95, e4, 79 ]
[ e7, c8, 37, 6d, 8d, d5, 4e, a9, 6c, 56, f4, ea, 65, 7a, ae, 08 ]
[ ba, 78, 25, 2e, 1c, a6, b4, c6, e8, dd, 74, 1f, 4b, bd, 8b, 8a ]
[ 70, 3e, b5, 66, 48, 03, f6, 0e, 61, 35, 57, b9, 86, c1, 1d, 9e ]
[ e1, f8, 98, 11, 69, d9, 8e, 94, 9b, 1e, 87, e9, ce, 55, 28, 

Die Umkehrung der S-Box

In [12]:
def aes_sbox_inv(x: Poly) -> Poly:
    "AES inverse S-box."
    res = x.coeff
    res += [0] * (8 - len(res))
    res = PolyAES(aes_A_inv * (Matrix(res) - aes_b))
    if res:
        res = res.inv()
    return res

Matrix([[ format(poly2byte(aes_sbox_inv(byte2poly(i + 16 * j))), '02x') for i in range(16)] for j in range(16)])

[ 52, 09, 6a, d5, 30, 36, a5, 38, bf, 40, a3, 9e, 81, f3, d7, fb ]
[ 7c, e3, 39, 82, 9b, 2f, ff, 87, 34, 8e, 43, 44, c4, de, e9, cb ]
[ 54, 7b, 94, 32, a6, c2, 23, 3d, ee, 4c, 95, 0b, 42, fa, c3, 4e ]
[ 08, 2e, a1, 66, 28, d9, 24, b2, 76, 5b, a2, 49, 6d, 8b, d1, 25 ]
[ 72, f8, f6, 64, 86, 68, 98, 16, d4, a4, 5c, cc, 5d, 65, b6, 92 ]
[ 6c, 70, 48, 50, fd, ed, b9, da, 5e, 15, 46, 57, a7, 8d, 9d, 84 ]
[ 90, d8, ab, 00, 8c, bc, d3, 0a, f7, e4, 58, 05, b8, b3, 45, 06 ]
[ d0, 2c, 1e, 8f, ca, 3f, 0f, 02, c1, af, bd, 03, 01, 13, 8a, 6b ]
[ 3a, 91, 11, 41, 4f, 67, dc, ea, 97, f2, cf, ce, f0, b4, e6, 73 ]
[ 96, ac, 74, 22, e7, ad, 35, 85, e2, f9, 37, e8, 1c, 75, df, 6e ]
[ 47, f1, 1a, 71, 1d, 29, c5, 89, 6f, b7, 62, 0e, aa, 18, be, 1b ]
[ fc, 56, 3e, 4b, c6, d2, 79, 20, 9a, db, c0, fe, 78, cd, 5a, f4 ]
[ 1f, dd, a8, 33, 88, 07, c7, 31, b1, 12, 10, 59, 27, 80, ec, 5f ]
[ 60, 51, 7f, a9, 19, b5, 4a, 0d, 2d, e5, 7a, 9f, 93, c9, 9c, ef ]
[ a0, e0, 3b, 4d, ae, 2a, f5, b0, c8, eb, bb, 3c, 83, 53, 99, 

Alternativ als Substitutionstabellen:

In [13]:
S = [ poly2byte(aes_sbox(byte2poly(i))) for i in range(256) ]
Si = [ 0 for i in range(256) ]
for i in range(256):
    Si[S[i]] = i

Si[S[12]]

12

Der AES-Körper zusammen mit der S-Box ist auch im Paket verfügbar:

In [14]:
from kryptools import GF2_aes

x = 3
assert int(GF2_aes(x) ** -1) == poly2byte(byte2poly(x).inv())
assert int(GF2_aes(x).sbox()) == poly2byte(aes_sbox(byte2poly(x)))

Die S-Box kann übrigens auch als Polynom dargestellt werden:

In [15]:
from kryptools import lagrange_interpolation

x = list(GF2_aes)
y = [a.sbox() for a in x]
lagrange_interpolation(x, y)

05 x^254 + 09 x^253 + f9 x^251 + 25 x^247 + f4 x^239 + x^223 + b5 x^191 + 8f x^127 + 63

In [16]:
x = list(GF2_aes)
y = [GF2_aes(0)] + [a**-1 for a in GF2_aes.star()]
lagrange_interpolation(x, y)

x^254

Damit können wir nun einen AES-Zustand als Matrix schreiben und die einzelnen Transformationen anwenden:

In [17]:
s = b'\x02' + b'\x00' * 15
state = zeros(4)
for l in range(16):
    i, j = divmod(l, 4)
    state[j, i] = byte2poly(s[l])

state

[ x, 0, 0, 0 ]
[ 0, 0, 0, 0 ]
[ 0, 0, 0, 0 ]
[ 0, 0, 0, 0 ]

Alternativ als Bytes:

In [18]:
state.applyfunc(poly2byte)

[ 2, 0, 0, 0 ]
[ 0, 0, 0, 0 ]
[ 0, 0, 0, 0 ]
[ 0, 0, 0, 0 ]

Anwenden von `SubBytes`:

In [19]:
state.map(aes_sbox)
state

[ x^6 + x^5 + x^4 + x^2 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[             x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[             x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[             x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]

Anwenden von `ShiftRows`

In [20]:
for i in range(state.rows):
    state.matrix[i] = rotate(state.matrix[i], i)

state

[ x^6 + x^5 + x^4 + x^2 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[             x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[             x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[             x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]

und `MixColumns`

In [21]:
aes_mix_matrix = Matrix([ rotate([2, 3, 1, 1], -n) for n in range(4) ], ring=byte2poly)

state.matrix = (aes_mix_matrix * state).matrix
state

[             x^6 + x^3 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[ x^6 + x^5 + x^4 + x^2 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[ x^6 + x^5 + x^4 + x^2 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]
[ x^6 + x^4 + x^3 + x^2 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1, x^6 + x^5 + x + 1 ]

Damit sind alle Basisoperationen des AES implementiert. Unser Paket stellt für den AES zwei Klassen zur Verfügung: eine, die der AES-Zustandsmatrix entspricht und die notwendigen Manipulationen zur Verfügung stellt,
und eine zweite, die den Schlüssel als Matrix speichert, aus der dann die Rundenschlüssel generiert werden können.

In [22]:
from kryptools import AESKeySchedule, AESCipher

### Test-Vektoren aus FIPS-197

In [23]:
key = 0x2b7e151628aed2a6abf7158809cf4f3c.to_bytes(16, 'big')
key = AESKeySchedule(key)
key.showtmp=True

print(key)
print("------------------")
for _ in range(2):
    print(f"----Round {key.round+1}-------")
    key.next()
    print(key)

[ 2b, 28, ab, 09 ]
[ 7e, ae, f7, cf ]
[ 15, d2, 15, 4f ]
[ 16, a6, 88, 3c ]
------------------
----Round 1-------
Start    :[09, cf, 4f, 3c]
RotWord  :[cf, 4f, 3c, 09]
SubWord  :[8a, 84, eb, 01]
AddConst :[8b, 84, eb, 01]
[ a0, 88, 23, 2a ]
[ fa, 54, a3, 6c ]
[ fe, 2c, 39, 76 ]
[ 17, b1, 39, 05 ]
----Round 2-------
Start    :[2a, 6c, 76, 05]
RotWord  :[6c, 76, 05, 2a]
SubWord  :[50, 38, 6b, e5]
AddConst :[52, 38, 6b, e5]
[ f2, 7a, 59, 73 ]
[ c2, 96, 35, 59 ]
[ 95, b9, 80, f6 ]
[ f2, 43, 7a, 7f ]


In [24]:
key = 0x2b7e151628aed2a6abf7158809cf4f3c.to_bytes(16, 'big')
s = 0x3243f6a8885a308d313198a2e0370734.to_bytes(16, 'big')

key = AESKeySchedule(key)
state = AESCipher()
state(s)

print(state)
print("------------------")
print("----AddKey--------")
state.add_key(key.key)
print(state)
for _ in range(1):
    key.next()
    print(f"----Round {key.round}-------")
    print("----SubBytes------")
    state.sub_bytes()
    print(state)
    print("----ShiftRows-----")
    state.shift_rows()
    print(state)
    print("----MixColumns----")
    state.mix_columns()
    print(state)
    print("----AddKey--------")
    state.add_key(key.key)
    print(state)


[ 32, 88, 31, e0 ]
[ 43, 5a, 31, 37 ]
[ f6, 30, 98, 07 ]
[ a8, 8d, a2, 34 ]
------------------
----AddKey--------
[ 19, a0, 9a, e9 ]
[ 3d, f4, c6, f8 ]
[ e3, e2, 8d, 48 ]
[ be, 2b, 2a, 08 ]
----Round 1-------
----SubBytes------
[ d4, e0, b8, 1e ]
[ 27, bf, b4, 41 ]
[ 11, 98, 5d, 52 ]
[ ae, f1, e5, 30 ]
----ShiftRows-----
[ d4, e0, b8, 1e ]
[ bf, b4, 41, 27 ]
[ 5d, 52, 11, 98 ]
[ 30, ae, f1, e5 ]
----MixColumns----
[ 04, e0, 48, 28 ]
[ 66, cb, f8, 06 ]
[ 81, 19, d3, 26 ]
[ e5, 9a, 7a, 4c ]
----AddKey--------
[ a4, 68, 6b, 02 ]
[ 9c, 9f, 5b, 6a ]
[ 7f, 35, ea, 50 ]
[ f2, 2b, 43, 49 ]


### Beispiele aus dem Skriptum

In [25]:
state = AESCipher()
state(b'\x02' + b'\x00' * 15)
state.sub_bytes()
state

[ 77, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]

In [26]:
state([[0xAB, 0xD5, 0x04, 0x88], [0xAB, 0xD5, 0x04, 0x88], [0xAB, 0xD5, 0x04, 0x88], [0xAB, 0xD5, 0x04, 0x88]])
state.shift_rows()
state

[ ab, d5, 04, 88 ]
[ d5, 04, 88, ab ]
[ 04, 88, ab, d5 ]
[ 88, ab, d5, 04 ]

In [27]:
s = [ [ 0 ] * 4 for _ in range(4) ]
s[1][0] = 0x01
s[3][0] = 0x02
state(s)
state.mix_columns()
state

[ 01, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
[ 07, 00, 00, 00 ]
[ 05, 00, 00, 00 ]

In [28]:
key = AESKeySchedule(b'\x00' * 12 + b'\x01' + b'\x00' * 3)
state.add_key(key.key)
state

[ 01, 00, 00, 01 ]
[ 00, 00, 00, 00 ]
[ 07, 00, 00, 00 ]
[ 05, 00, 00, 00 ]

In [29]:
key.next()
key

[ 62, 62, 62, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 7c, 7c, 7c, 7c ]

### Diffusion

Unterschiede im AES-Zustand für zwei Zustände, die sich im ersten Eintrag um ein Bit unterscheiden.

In [30]:
(b'\x01' + b'\x00' * 15).hex()

'01000000000000000000000000000000'

In [31]:
key = 0x2b7e151628aed2a6abf7158809cf4f3c.to_bytes(16, 'big')
key = AESKeySchedule(key)
state0 = AESCipher()
state0(b'\x01' + b'\x00' * 15)
state1 = AESCipher()
state1(b'\x00' + b'\x00' * 15)

for state in [ state0, state1 ]:
    state.add_key(key.key)

for i in range(1,4):
    print(f"Round {i}: after SubByte")
    for state in [ state0, state1 ]:
        state.sub_bytes()
    print((state0.state + state1.state))

    print(f"Round {i}: after ShiftRows")
    for state in [ state0, state1 ]:
        state.shift_rows()
    print((state0.state + state1.state))

    print(f"Round {i}: after MixColums")
    key.next()
    for state in [ state0, state1 ]:
        state.mix_columns()
        state.add_key(key.key)
    print((state0.state + state1.state))


Round 1: after SubByte
[ 14, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
Round 1: after ShiftRows
[ 14, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
Round 1: after MixColums
[ 28, 00, 00, 00 ]
[ 14, 00, 00, 00 ]
[ 14, 00, 00, 00 ]
[ 3c, 00, 00, 00 ]
Round 2: after SubByte
[ 39, 00, 00, 00 ]
[ 2a, 00, 00, 00 ]
[ ad, 00, 00, 00 ]
[ 45, 00, 00, 00 ]
Round 2: after ShiftRows
[ 39, 00, 00, 00 ]
[ 00, 00, 00, 2a ]
[ 00, 00, ad, 00 ]
[ 00, 45, 00, 00 ]
Round 2: after MixColums
[ 72, 45, ad, 7e ]
[ 39, 45, ec, 54 ]
[ 39, cf, 41, 2a ]
[ 4b, 8a, ad, 2a ]
Round 3: after SubByte
[ c5, 67, b1, 5d ]
[ fb, eb, 37, 66 ]
[ f2, fd, d4, d1 ]
[ 2d, 01, fc, 9e ]
Round 3: after ShiftRows
[ c5, 67, b1, 5d ]
[ eb, 37, 66, fb ]
[ d4, d1, f2, fd ]
[ 9e, 2d, 01, fc ]
Round 3: after MixColums
[ fd, 6b, 20, ad ]
[ f1, 4c, 71, 50 ]
[ 24, 9e, 2b, 58 ]
[ 4c, 15, 5e, 02 ]


Null-Zustand mit Null-Schlüssel

In [32]:
key = AESKeySchedule(b'\x00' * 16)
state = AESCipher()
state(b'\x00' * 16)


print(f"After AddKey")
state.add_key(key.key)
print(state.state)

for i in range(1,4):
    print(f"Round {i}: after SubByte")
    state.sub_bytes()
    print(state)

    print(f"Round {i}: after ShiftRows")
    state.shift_rows()
    print(state)

    print(f"Round {i}: after MixColums")
    state.mix_columns()
    print(state)

    print(f"Round {i}: after AddKey")
    key.next()
    state.mix_columns()
    state.add_key(key.key)
    print(state)

After AddKey
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
Round 1: after SubByte
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
Round 1: after ShiftRows
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
Round 1: after MixColums
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
Round 1: after AddKey
[ 01, 01, 01, 01 ]
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
[ 00, 00, 00, 00 ]
Round 2: after SubByte
[ 7c, 7c, 7c, 7c ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
Round 2: after ShiftRows
[ 7c, 7c, 7c, 7c ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
[ 63, 63, 63, 63 ]
Round 2: after MixColums
[ 5d, 5d, 5d, 5d ]
[ 7c, 7c, 7c, 7c ]
[ 7c, 7c, 7c, 7c ]
[ 42, 42, 42, 42 ]
Round 2: after AddKey
[ 9b, f9, 9b, f9 ]
[ fb, 98, fb, 98 ]
[ 87, e4, 87, e4 ]
[ aa, c9, aa, c9 ]
Round 3: after SubByte
[ 14, 99, 14, 99 ]
[ 0f, 46, 0f, 46 ]
[ 17, 69, 17, 69 ]
[ ac, dd, ac, dd ]
Round 3: after

### Betriebsmodi

Im Paket gibt es eine Klasse, die uns ausgehend von einer Blockchiffre die verschiedenen Betriebsmodi (ECB, CBC etc.) implementiert. Ist Padding notwendig, so wird PKCS#7 verwendet. Ein notwendiger IV wird automatisch gewählt und als erster Geheimtextblock ausgegeben.

In [33]:
from kryptools import DESBlockCipher

key = b"PASSWORT"
text = b"Eine spannende Testnachricht auf die Sie schon lange gewartet haben."

des = DESBlockCipher(key)
ctext = des.encrypt_cbc(text)
assert des.decrypt_cbc(ctext) == text
ctext

b'\xcf\xb0~\x03\x18\xe0\xf7\xefP\xc8\x9dO\x87\xd3D\x98\xc4\xc7\x05P\xd2\xf3\x05\x97j\x8fD\xcf\x06\xcch\x94\xe4I\x83\xcc\x7f\x8a\xc4Dt_\x12\x00\xca\x92x\xd01Cr\x98\xf5\xa8I\xf1\xfe\xeeB\xf3#\xc8\x02A\xa7\x0c\xbd@%K(\x10\xd6\xc6:\xe4\xac\td\xff'

Verschlüsselung mit AES im GCM-Modus

In [34]:
from kryptools import BlockCipher, AESBlockCipher

key = b'0123456789ABCDEF'
aad = b'Vorweg eine ungeheim autenthische Nachricht.'
text = b'Und jetzt der streng geheime Geheimtext.'

aes = AESBlockCipher(key)
ctext = aes.encrypt_gcm(text, aad=aad)
assert aes.decrypt_gcm(ctext, aad=aad) == text
aad, ctext

(b'Vorweg eine ungeheim autenthische Nachricht.',
 b'\x93\xf5%1\xa4\\\xdb\x95V\xb7\xd3\\f)\xf4V9\xc6]\xfe-\xb4\x91AYz\x99\xfa\xa9O\xc5\xeb\x91*\xff\x8b]\x1a\xb2[\xefIW|\xf0%\x81q\xab\xa6S\xf6\x1d\xea\xb7\xbd\x93\xdeV\x06\\\x11c\xb3\x8aL\xfd\xd1')

## MACs

Berechnung der vereinfachten CBC-MAC-Variante aus dem Buch:

In [35]:
def str2letters(msg: str) -> list[int]:
    return [ (i - 65) % 26 for i in msg.upper().encode()]

pt = str2letters("FAD")
pt

[5, 0, 3]

Nur MAC vom Klartext:

In [36]:
def cbc(msg:list, iv: int, k: int) -> list:
    y = iv
    out = [ y ]
    for x in msg:
        y = (x + y + k) % 26
        out.append(y)
    return out

h, k = 5, 3

cbc(pt, 0, h)[-1]

23

Geheimtext plus MAC vom Geheimtext:

In [37]:
ct = cbc(pt, 10, k)
ct.append(cbc(ct, 0, h)[-1])
ct

[10, 18, 21, 1, 18]